# 第21章 特征工程与数据预处理

这个 notebook 用一个小型调查数据说明缺失值处理、标准化、类别编码、特征构造和特征选择如何直接改变模型表现。

## 学习目标

- 理解预处理步骤为什么会改变模型表现
- 区分有物理意义的特征和可能带来误导的杂讯特征
- 练习训练集统计量驱动的缺失值填补和标准化
- 理解 one-hot 编码和特征构造的基本作用
- 看到数据泄漏是如何悄悄发生的

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from pathlib import Path

DATA_PATH = Path('../../data/small/object_preprocessing_demo.csv')
with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))

numeric_keys = ('u_g', 'g_r', 'r_i', 'seeing_arcsec', 'detector_x')
for row in rows:
    for key in numeric_keys:
        row[key] = float(row[key]) if row[key] != '' else None

print('loaded rows:', len(rows))
print('classes:', sorted({row['object_class'] for row in rows}))
print('Python version:', platform.python_version())


In [ ]:
test_indices = {1, 4, 5, 8, 11, 14}
train_rows = [row.copy() for index, row in enumerate(rows) if index not in test_indices]
test_rows = [row.copy() for index, row in enumerate(rows) if index in test_indices]

print('train:', len(train_rows), 'test:', len(test_rows))
print('test ids:', [row['object_id'] for row in test_rows])


## 先看数据问题：缺失值与量纲差异

这里我们刻意加入了缺失值和一个量级很大的仪器坐标特征 `detector_x`。

In [ ]:
missing_summary = {}
for key in ('seeing_arcsec', 'sky_region'):
    missing_summary[key] = sum(1 for row in rows if row[key] in (None, ''))

range_summary = {}
for key in numeric_keys:
    values = [row[key] for row in rows if row[key] is not None]
    range_summary[key] = (round(min(values), 3), round(max(values), 3))

print('missing summary =', missing_summary)
print('range summary =', range_summary)


## 用训练集统计量做最小预处理

注意：均值和众数都只能从训练集计算。

In [ ]:
def numeric_mean(rows_subset, key):
    values = [row[key] for row in rows_subset if row[key] is not None]
    return sum(values) / len(values)

numeric_means = {key: numeric_mean(train_rows, key) for key in numeric_keys}
region_candidates = [row['sky_region'] for row in train_rows if row['sky_region'] not in (None, '')]
region_mode = max(sorted(set(region_candidates)), key=region_candidates.count)

def apply_basic_imputation(rows_subset):
    processed = []
    for row in rows_subset:
        item = row.copy()
        for key in numeric_keys:
            if item[key] is None:
                item[key] = numeric_means[key]
        if item['sky_region'] in (None, ''):
            item['sky_region'] = region_mode
        processed.append(item)
    return processed

train_imputed = apply_basic_imputation(train_rows)
test_imputed = apply_basic_imputation(test_rows)

print('numeric means =', {key: round(value, 3) for key, value in numeric_means.items()})
print('region mode =', region_mode)


## 一个最小最近中心分类器

我们故意保持模型简单，把注意力放在特征表示本身。

In [ ]:
def centroid(vectors):
    return [sum(vector[index] for vector in vectors) / len(vectors) for index in range(len(vectors[0]))]

def euclidean_distance(a_value, b_value):
    return math.sqrt(sum((ax - bx) ** 2 for ax, bx in zip(a_value, b_value)))

def nearest_centroid_predict(train_subset, test_subset, feature_fn):
    grouped = {}
    for row in train_subset:
        grouped.setdefault(row['object_class'], []).append(feature_fn(row))
    centroids = {label: centroid(vectors) for label, vectors in grouped.items()}

    predictions = []
    for row in test_subset:
        features = feature_fn(row)
        best_label = min(
            centroids,
            key=lambda label: euclidean_distance(features, centroids[label]),
        )
        predictions.append(best_label)
    return predictions

def accuracy_score(y_true, y_pred):
    return sum(1 for truth, pred in zip(y_true, y_pred) if truth == pred) / len(y_true)

truths = [row['object_class'] for row in test_imputed]


## 对照 1：只做最小填补，直接使用原始数值列

这里 `detector_x` 的量级远大于颜色指标，因此距离会被它主导。

In [ ]:
def raw_feature_vector(row):
    return [row[key] for key in numeric_keys]

raw_predictions = nearest_centroid_predict(train_imputed, test_imputed, raw_feature_vector)
raw_accuracy = accuracy_score(truths, raw_predictions)
print('raw accuracy =', round(raw_accuracy, 3))
print(list(zip([row['object_id'] for row in test_imputed], truths, raw_predictions)))


## 一个故意错误的泄漏示例

下面这一步故意用全数据均值做填补，只是为了展示它和正确做法的不同。不要把它当作正确流程。

In [ ]:
leaky_means = {}
for key in numeric_keys:
    values = [row[key] for row in rows if row[key] is not None]
    leaky_means[key] = sum(values) / len(values)

print('train-only mean seeing_arcsec =', round(numeric_means['seeing_arcsec'], 3))
print('all-data mean seeing_arcsec =', round(leaky_means['seeing_arcsec'], 3))


## 对照 2：对数值特征做标准化

即使模型不变，只要把不同量纲拉到可比较的尺度，结果就会显著变化。

In [ ]:
numeric_stats = {}
for key in numeric_keys:
    values = [row[key] for row in train_imputed]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    numeric_stats[key] = (mean_value, math.sqrt(variance) or 1.0)

def scaled_feature_vector(row):
    return [
        (row[key] - numeric_stats[key][0]) / numeric_stats[key][1]
        for key in numeric_keys
    ]

scaled_predictions = nearest_centroid_predict(train_imputed, test_imputed, scaled_feature_vector)
scaled_accuracy = accuracy_score(truths, scaled_predictions)
print('scaled accuracy =', round(scaled_accuracy, 3))
print(list(zip([row['object_id'] for row in test_imputed], truths, scaled_predictions)))


## 对照 3：做特征工程和类别编码

这里我们丢掉明显更像仪器坐标的 `detector_x`，保留更接近物理信息的颜色特征，构造 `color_sum`，并对 `sky_region` 做 one-hot 编码。

In [ ]:
base_keys = ('u_g', 'g_r', 'r_i', 'seeing_arcsec')
base_stats = {}
for key in base_keys:
    values = [row[key] for row in train_imputed]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    base_stats[key] = (mean_value, math.sqrt(variance) or 1.0)

color_sum_values = [row['u_g'] + row['g_r'] + row['r_i'] for row in train_imputed]
color_sum_mean = sum(color_sum_values) / len(color_sum_values)
color_sum_std = math.sqrt(sum((value - color_sum_mean) ** 2 for value in color_sum_values) / len(color_sum_values)) or 1.0
region_levels = ['equator', 'north', 'south']

def engineered_feature_vector(row):
    values = [
        (row[key] - base_stats[key][0]) / base_stats[key][1]
        for key in base_keys
    ]
    color_sum = row['u_g'] + row['g_r'] + row['r_i']
    values.append((color_sum - color_sum_mean) / color_sum_std)
    values.extend(1.0 if row['sky_region'] == level else 0.0 for level in region_levels)
    return values

engineered_predictions = nearest_centroid_predict(train_imputed, test_imputed, engineered_feature_vector)
engineered_accuracy = accuracy_score(truths, engineered_predictions)
print('engineered accuracy =', round(engineered_accuracy, 3))
print(list(zip([row['object_id'] for row in test_imputed], truths, engineered_predictions)))


In [ ]:
comparison = {
    'raw': round(raw_accuracy, 3),
    'scaled': round(scaled_accuracy, 3),
    'engineered': round(engineered_accuracy, 3),
}
comparison


## 如果安装了 scikit-learn

下面这个单元是可选的。它把手写流程整理成更接近真实项目的 `ColumnTransformer + Pipeline` 结构。

In [ ]:
try:
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
except ModuleNotFoundError:
    print('scikit-learn 未安装；已跳过标准库预处理示例。')
else:
    x_train = [
        {
            'u_g': row['u_g'],
            'g_r': row['g_r'],
            'r_i': row['r_i'],
            'seeing_arcsec': row['seeing_arcsec'],
            'sky_region': row['sky_region'],
        }
        for row in train_rows
    ]
    x_test = [
        {
            'u_g': row['u_g'],
            'g_r': row['g_r'],
            'r_i': row['r_i'],
            'seeing_arcsec': row['seeing_arcsec'],
            'sky_region': row['sky_region'],
        }
        for row in test_rows
    ]
    y_train = [row['object_class'] for row in train_rows]
    y_test = [row['object_class'] for row in test_rows]

    print('Optional sklearn pipeline scaffold prepared for x_train/x_test/y_train/y_test.')


## Feature Ledger Export

这一段导出最小 Feature Ledger。它回答：模型看见了哪些特征、这些特征从哪里来、哪些统计量只在训练集上拟合、是否存在泄漏或选择效应风险。

In [ ]:
RESULTS_DIR = Path('results/part3_evidence_artifacts')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

feature_ledger_rows = [
    {
        'feature': name,
        'raw_or_derived': 'raw scaled',
        'source_field': name,
        'unit': 'mag/color or arcsec' if name == 'seeing_arcsec' else 'mag/color',
        'fit_on_train_only': 'yes: imputation mean and scaler',
        'leakage_risk': 'low if train-only statistics are preserved',
        'keep_drop_reason': 'kept for engineered nearest-centroid model',
    }
    for name in base_keys
]
feature_ledger_rows.append({
    'feature': 'color_sum',
    'raw_or_derived': 'derived',
    'source_field': 'u_g + g_r + r_i',
    'unit': 'mag/color sum',
    'fit_on_train_only': 'yes: scaling mean/std from training set',
    'leakage_risk': 'low; derived only from input colors',
    'keep_drop_reason': 'kept as a compact color-shape feature',
})
for level in region_levels:
    feature_ledger_rows.append({
        'feature': f'sky_region_{level}',
        'raw_or_derived': 'encoded categorical',
        'source_field': 'sky_region',
        'unit': 'binary indicator',
        'fit_on_train_only': 'yes: missing category mode from training set',
        'leakage_risk': 'medium; sky region may encode selection effects',
        'keep_drop_reason': 'kept to demonstrate categorical encoding audit',
    })
feature_ledger_rows.append({
    'feature': 'detector_x',
    'raw_or_derived': 'raw',
    'source_field': 'detector_x',
    'unit': 'detector coordinate',
    'fit_on_train_only': 'n/a after drop',
    'leakage_risk': 'medium/high; may encode instrument or observing pattern rather than astrophysics',
    'keep_drop_reason': 'dropped from engineered model after raw-scale diagnostic',
})

headers = ['feature', 'raw_or_derived', 'source_field', 'unit', 'fit_on_train_only', 'leakage_risk', 'keep_drop_reason']
ledger_lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for row in feature_ledger_rows:
    ledger_lines.append('| ' + ' | '.join(str(row[column]) for column in headers) + ' |')

artifact = f'''# Ch21 Feature Ledger

## Preprocessing Summary

- Dataset: {DATA_PATH.name}
- Split: fixed teaching train/test split; preprocessing statistics fitted on training rows only.
- Missing numeric values: filled with training-set means.
- Missing sky_region: filled with training-set mode `{region_mode}`.
- Scaling: numeric means/stds from training rows only.
- Accuracy comparison: raw={raw_accuracy:.3f}, scaled={scaled_accuracy:.3f}, engineered={engineered_accuracy:.3f}.
- Leakage demonstration: all-data seeing mean={leaky_means['seeing_arcsec']:.3f}; train-only seeing mean={numeric_means['seeing_arcsec']:.3f}.

## Feature Ledger

{chr(10).join(ledger_lines)}

## Claim Boundary

This ledger supports a preprocessing audit for a tiny teaching classifier. It cannot prove that sky_region or detector-related fields are scientifically causal features.
'''

output_path = RESULTS_DIR / 'ch21_feature_ledger.md'
output_path.write_text(artifact, encoding='utf-8')
print('wrote', output_path)
print('feature ledger rows:', len(feature_ledger_rows))


## 小结

这个例子说明：预处理不是模型前面的清洁工，而是在重新定义模型所看到的特征空间。标准化、编码、特征构造和特征剔除，都会直接改变模型边界。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'raw_accuracy': round(raw_accuracy, 3),
    'scaled_accuracy': round(scaled_accuracy, 3),
    'engineered_accuracy': round(engineered_accuracy, 3),
    'python_version': platform.python_version(),
}

print('Preprocessing snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
